Elena Osipyan

1010219381

Assignment 3, CTA200

In [ ]:
#imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import comp_calc

Question 1

In [ ]:
#grids
N = 100
x = np.linspace(-2, 2, N)
y = np.linspace(-2, 2, N)

sets = np.zeros((N, N))

#compute
for i in range(N):
    for j in range(N):
        c = x[i] + 1j * y[j]
        sets[j, i] = comp_calc.comp_calc(c, max_iter=30)

#plot 1: bounded vs diverged
plt.figure()
plt.imshow(sets==30, extent=(-2,2,-2,2), cmap='magma') # sets set to 30, as thats the max
plt.title("Bounded vs Diverged")
plt.xlabel("Re(c)")
plt.ylabel("Im(c)")
plt.savefig('mandelbrot_bounded.pdf', bbox_inches='tight', dpi=300)
plt.show()

#plot 2: iterations
plt.figure()
plt.imshow(sets, extent=(-2,2,-2,2), cmap='magma')
plt.title("Iterations")
plt.xlabel("Re(c)")
plt.ylabel("Im(c)")
plt.colorbar(label="Iteration to divergence")
plt.savefig('mandelbrot_iterations.pdf', bbox_inches='tight', dpi=300)
plt.show()



Q2, P1

In [ ]:
def lorenz_equations(t, W, sigma, r, b):
    """
    Defines the Lorenz equations.

    Parameters:
    -----------
    t : float
        Time (dimensionless)
    W : array-like
        State vector [X, Y, Z]
    sigma : float
        Prandtl number (kinematic viscosity / thermal diffusivity)
    r : float
        Rayleigh number (normalized temperature difference)
    b : float
        Dimensionless length scale

    Returns:
    --------
    dW : ndarray
        Time derivatives [dX/dt, dY/dt, dZ/dt]
    """
    X, Y, Z = W
    dX = -sigma * (X - Y)
    dY = r * X - Y - X * Z
    dZ = -b * Z + X * Y
    return np.array([dX, dY, dZ])


def integrate_lorenz(W0, t_max, sigma, r, b, t_eval=None, dense=False):
    """
    Integrate the Lorenz equations using solve_ivp.

    Parameters:
    -----------
    W0 : array-like
        Initial conditions [X0, Y0, Z0]
    t_max : float
        Final integration time
    sigma, r, b : float
        Lorenz parameters
    t_eval : ndarray, optional
        Time points for output
    dense : bool
        If True, request dense output for interpolation

    Returns:
    --------
    sol : OdeResult
        Solution object from solve_ivp
    """
    t_span = (0, t_max)
    sol = solve_ivp(lorenz_equations, t_span, W0, args=(sigma, r, b), method='RK45', t_eval=t_eval, dense_output=dense, max_step=0.01)
    #using RK45 b/c of 5th order accuracy
    return sol



Q2, P2-P4

In [ ]:
#set the initial conditions
W0 = np.array([0., 1., 0.])
t_max = 60

# set parameter values for sigma, r, b
sigma = 10.
r = 28.
b = 8./3.

#evaluation values for t (6000 steps because Lorenz performed 6000 iterations)
t_eval = np.linspace(0, t_max, 6000)

#integrate
sol = integrate_lorenz(W0, t_max, sigma=sigma, r=r, b=b, t_eval=t_eval)

print(sol)

#y values are the second row of sol
Y = sol.y[1]
time = sol.t

#plotting in 3 subplots (each 1000 steps)
fig, ax = plt.subplots(3, 1, figsize=(6, 6), sharey=True)
fig.patch.set_facecolor('white')

for i in range(3):
    start = i * 1000
    end = (i + 1) * 1000
    step_range = np.arange(start, end)
    ax[i].plot(step_range, Y[start:end], color='purple', linewidth=1.5)
    ax[i].set_xlabel("Iterations")
    ax[i].set_ylabel("Y")
    ax[i].set_facecolor('white')
    ax[i].spines['top'].set_color('black')
    ax[i].spines['bottom'].set_color('black')
    ax[i].spines['left'].set_color('black')
    ax[i].spines['right'].set_color('black')
    ax[i].tick_params(colors='black')
    ax[i].xaxis.label.set_color('black')
    ax[i].yaxis.label.set_color('black')

fig.suptitle("Graph of Y as a Function of Time, First 3000 Iterations", color='black', fontsize=12)
plt.tight_layout()
plt.savefig('lorenz_timeseries.pdf', bbox_inches='tight', dpi=300)
plt.show()

# getting data for iterations 1400-1900
X_segment = sol.y[0][1400:1901]
Y_segment = sol.y[1][1400:1901]
Z_segment = sol.y[2][1400:1901]

fig, ax = plt.subplots(2, 1, figsize=(6, 12))
fig.patch.set_facecolor('white')

# plot YZ plane projection
ax[0].plot(Y_segment, Z_segment, color='purple', linewidth=1.5)
ax[0].set_xlabel("Y")
ax[0].set_ylabel("Z")
ax[0].set_title("Projection on YZ Plane of Phase Space")
ax[0].set_facecolor('white')
ax[0].spines['top'].set_color('black')
ax[0].spines['bottom'].set_color('black')
ax[0].spines['left'].set_color('black')
ax[0].spines['right'].set_color('black')
ax[0].tick_params(colors='black')
ax[0].xaxis.label.set_color('black')
ax[0].yaxis.label.set_color('black')

#plot XY plane projection
ax[1].plot(Y_segment, X_segment, color='purple', linewidth=1.5)
ax[1].invert_yaxis()
ax[1].set_xlabel("Y")
ax[1].set_ylabel("X")
ax[1].set_title("Projection on XY Plane of Phase Space")
ax[1].set_facecolor('white')
ax[1].spines['top'].set_color('black')
ax[1].spines['bottom'].set_color('black')
ax[1].spines['left'].set_color('black')
ax[1].spines['right'].set_color('black')
ax[1].tick_params(colors='black')
ax[1].xaxis.label.set_color('black')
ax[1].yaxis.label.set_color('black')

#add labels
for iteration in range(1400, 1901, 100):
    label = iteration - 1400
    ax[0].text(Y_segment[label], Z_segment[label], str(iteration // 100), fontsize=7, color='black')
    ax[1].text(Y_segment[label], X_segment[label], str(iteration // 100), fontsize=7, color='black')

plt.subplots_adjust(hspace=0.3)
plt.savefig('lorenz_phase_yz.pdf', bbox_inches='tight', dpi=300)
plt.savefig('lorenz_phase_xy.pdf', bbox_inches='tight', dpi=300)
plt.show()

Q2, P5

In [ ]:
#slightly modified initial conditions
W0_perturbed = W0 + np.array([0., 1.e-8, 0.])

#integrate the ode, all other values shared with W0
sol_perturbed = integrate_lorenz(W0_perturbed, t_max, sigma=sigma, r=r, b=b, t_eval=t_eval)

distance = np.linalg.norm(sol_perturbed.y - sol.y, axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('white')
ax.semilogy(sol.t, distance, color='purple', linewidth=1.5)
ax.set_xlabel("Time t", color='black')
ax.set_ylabel("Distance between W(t) and W'(t) (log scale)", color='black')
ax.set_title("Difference between Solutions to Lorenz Equations vs. Time", color='black')
ax.grid(True, color='black', alpha=0.3)
ax.set_facecolor('white')
ax.spines['top'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.spines['left'].set_color('black')
ax.spines['right'].set_color('black')
ax.tick_params(colors='black')
plt.savefig('lorenz_divergence.pdf', bbox_inches='tight', dpi=300)
plt.show()
